In [1]:
import os
import pickle
import pandas as pd
from time import time
from typing import List, Optional
from collections import defaultdict

from docling.document_converter import DocumentConverter
from langchain_core.documents import Document


def parse_bbox(bbox):
    """
    docling bbox 안전 파싱
    - bbox 길이가 4 이상이면 앞의 4개만 사용
    - dict 형태도 지원
    """
    if not bbox:
        return None

    if isinstance(bbox, dict):
        return (
            bbox.get("x0", 0),
            bbox.get("y0", 0),
            bbox.get("x1", 0),
            bbox.get("y1", 0),
        )

    if isinstance(bbox, (list, tuple)) and len(bbox) >= 4:
        return tuple(bbox[:4])

    return None


def parse_pdf_docling_page_ordered(
    file_path: str,
    lv1_cat: str,
    lv2_cat: str,
    save_dir: Optional[str] = None,
    save_pickle: bool = True,
) -> List[Document]:
    """
    Docling 기반 PDF 파싱 (최종 완성형)

    - doc.pages 는 int 페이지 번호로 처리
    - provenance(page_no, bbox) 기준으로 텍스트/테이블 분기
    - bbox(y, x) 기준 정렬로 페이지 내 순서 완벽 보존
    - 테이블: DataFrame → Markdown
    - 텍스트: docling 기본 파싱

    Returns:
        List[LangChain Document]
    """

    start_time = time()

    # --------------------
    # 기본 정보
    # --------------------
    path = file_path.replace("\\", "/")
    filename = os.path.basename(path)
    parsed_filename = filename.replace(".pdf", "")

    # --------------------
    # 문서 변환 (1회)
    # --------------------
    converter = DocumentConverter()
    conv_res = converter.convert(file_path)
    doc = conv_res.document

    # --------------------
    # 페이지별 인덱싱 (성능 최적화)
    # --------------------
    texts_by_page = defaultdict(list)
    for text in doc.texts:
        if text.prov:
            texts_by_page[text.prov[0].page_no].append(text)

    tables_by_page = defaultdict(list)
    for table in doc.tables:
        if table.prov:
            tables_by_page[table.prov[0].page_no].append(table)

    parsed_docs: List[Document] = []

    # --------------------
    # 페이지 단위 처리
    # --------------------
    for page_no in doc.pages:  # page_no: int
        page_items = []

        # ---- 텍스트 수집
        for text in texts_by_page.get(page_no, []):
            content = text.text.strip()
            if not content:
                continue

            bbox = parse_bbox(text.prov[0].bbox)
            if bbox:
                x0, y0, x1, y1 = bbox
                top_y = max(y0, y1)
            else:
                x0, top_y = 0, -1

            page_items.append({
                "type": "text",
                "x": x0,
                "top_y": top_y,
                "obj": text,
            })

        # ---- 테이블 수집
        for table in tables_by_page.get(page_no, []):

            bbox = parse_bbox(table.prov[0].bbox)
            if bbox:
                x0, y0, x1, y1 = bbox
                top_y = max(y0, y1)
            else:
                x0, top_y = 0, -1

            page_items.append({
                "type": "table",
                "x": x0,
                "top_y": top_y,
                "obj": table,
            })
        # --------------------
        # 페이지 내 순서 보존 정렬
        #   - 위 → 아래 : top_y 내림차순
        #   - 같은 줄 : 왼쪽 → 오른쪽
        # --------------------
        page_items.sort(key=lambda v: (-v["top_y"], v["x"]))

        # ---- 정렬된 순서대로 파싱
        for item in page_items:
            if item["type"] == "table":
                df: pd.DataFrame = item["obj"].export_to_dataframe(doc)
                content = df.to_markdown()
                block_type = "table"
            else:
                content = item["obj"].text
                block_type = "text"

            parsed_docs.append(
                Document(
                    page_content=content,
                    metadata={
                        "filename": filename,
                        "lv1_cat": lv1_cat,
                        "lv2_cat": lv2_cat,
                        "page": str(page_no),
                        "type": block_type,
                    },
                )
            )

    # --------------------
    # 결과 저장
    # --------------------
    if save_pickle:
        if save_dir is None:
            save_dir = f"./docs/{lv1_cat}_{lv2_cat}_parsed_r01"

        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{parsed_filename}.pkl")

        with open(save_path, "wb") as f:
            pickle.dump(parsed_docs, f)

    elapsed = time() - start_time
    print(
        f"✅ Parsed {filename} | "
        f"{len(parsed_docs)} blocks | "
        f"{elapsed:.2f}s"
    )

    return parsed_docs


d:\auto_vectordb\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = parse_pdf_docling_page_ordered(
    file_path="./file/kubota_sample.pdf",
    lv1_cat="CE",
    lv2_cat="KUBOTA",
    )

2026-01-18 12:47:48,073 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-01-18 12:47:48,297 - INFO - Going to convert document batch...
2026-01-18 12:47:48,301 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2026-01-18 12:47:48,370 - INFO - Loading plugin 'docling_defaults'
2026-01-18 12:47:48,377 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-01-18 12:47:48,424 - INFO - Loading plugin 'docling_defaults'
2026-01-18 12:47:48,439 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-01-18 12:47:48,477 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2026-01-18 12:47:48,703 - INFO - Accelerator device: 'cpu'
2026-01-18 12:47:51,007 - INFO - Auto OCR model selected easyocr.
2026-01-18 12:47:51,035 - INFO - Accelerator device: 'cpu'
2026-01-18 12:47:52,604 - INFO - Accelerator device: 'cpu'
2026-01-18 12:47:53,300 - INFO - Processing document kubota_sample.pdf
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\

✅ Parsed kubota_sample.pdf | 229 blocks | 188.21s


In [ ]:
# docs

In [3]:
merged_docs =dict()
for doc in docs:
    if doc.metadata["page"] not in list(merged_docs.keys()):
        merged_docs[doc.metadata["page"]] = doc.page_content
    else:
        if doc.metadata["type"] == 'text':
            merged_docs[doc.metadata["page"]] += "\n" + doc.page_content
        else:
            merged_docs[doc.metadata["page"]] += "\n\n" + doc.page_content

merged_docs

{'1': 'For Earth, For Life Kubota\nEngine output:\n13.3 PS / 9.8 kW\nMachine weight (Cabin / Canopy):\n1,470 / 1,420 kg\nKUBOTA MINI EXCAVATOR\n6',
 '2': "SUPERIOR PERFORMANCE/ DELUXE INTERIOR\nKubota's new KX015-4 mini excavator raises the standard in the 1.5-2.0 tonne category with a powerful digging force and a wider working range that rival higherend excavators. And with enhanced accessibility to sites such as roadsides or residential areas, the KX015-4 gets the jobs done easier from large construction areas to small places.\nKubota",
 '3': "Photo: KX016-4\nKubota\nEnhanced digging force\nThe KX015-4 delivers an impressive bucket digging force. Its powerful and well-balanced arm and bucket allow the operator to dig faster and more efficiently even in the toughest conditions.\nKubota original engine\nThe KX015-4 is powered by Kubota's impressive D782 13.1 PS engine. Engineered with the power to maximise digging and lifting performance, it also delivers minimised noise and vibration,

In [4]:
from IPython.display import Markdown
Markdown(merged_docs["6"])

SPECIFICATIONS
* 1  With 32.5 kg Kubota original bucket, full tanks, rubber shoe. 3.4 (0.35)
* 2   Machine weight with 75 kg operator. 3.0 (0.30) 1.5m
2.7 (0.27)
-
* 3   These values are mesured under specific conditions at maximum engine speed and can deviate, depending on the operating status. 3.6 (0.37) 2.5 (0.25) -4.5 (0.46) 1.0m
0.5m
3.4 (0.35)
3.3 (0.33)
2.3 (0.23)
2.2 (0.22)
2.6 (0.27)
-
Lifting point radius (max.)
Over-front
Blade UP
Over-side
-
-
1.7 (0.17)
-
5.4 (0.55)
5.3 (0.54) 0m LIFTING CAPACITY
-
-
1.1 (0.12)
-
*The lifting capacities are based on ISO 10567 and do not exceed 75% of the static tilt load of the machine or 87%
of the hydraulic lifting capacities of the machine.
**The excavator bucket, hook, sling and other lifting accessories are not included on this table.
★ All images shown are for brochure purposes only.
When operating the excavator, wear clothing and equipment in accordance to local legal and safety regulations.
KUBOTA (U.K.) LTD
Dormer Road,Thame, Oxfordshire, OX9 3UN, U.K.
Phone : 01844-268140
F a x   : 01844-216685
Axis of Rotation
Lift Point Radius
Lift Point
Lift Point Height
Lift Point Radius
Working ranges are with Kubota original bucket, without quick coupler. Axis of Rotation
* Specifications are subject to change without notice for purpose of improvement.
SEATBELT
2250
WORKING RANGE
3730
3790
990
1070
990
450
510
1490
KX15-4_
1070
Unit: mm
990
3360
2290
1810
2250
3790
3730
3710
1090
1450
1090
Axis of Rotation
Lift Point Radius
Lift Point
Lift Point Height
3360
2290
1810
2250
1490
240
230
2350
3790
3730
3710
1090
1450
1090
450
510
450
510
*With cabin, rubber shoe and standard arm kN (ton)
1070
1
990
990
K
KX15-4_
240
230
990
http://www.kubota-eu.com
Cat. No.8793-01-UK '23-03D. 2 2

|    | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.Machine weight *1 (cabin / canopy)                 | Model.kg                                                 | KX015-4.1470 / 1420               |
|---:|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:---------------------------------------------------------|:----------------------------------|
|  0 | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | Operating weight *2 (cabin / canopy)                     | kg                                                       | 1545 / 1495                       |
|  1 |                                                          | Model                                                    | Model                                                    | Model                                                    | Model                                                    | D782-E3-BH                        |
|  2 |                                                          | Type                                                     | Type                                                     | Type                                                     | Type                                                     | Water-cooled,diesel engine,E-TVCS |
|  3 |                                                          | Output ISO14396                                          | Output ISO14396                                          | Output ISO14396                                          | PS (kW)/rpm                                              | 13.3 (9.8) / 2300                 |
|  4 | Engine                                                   | Output ISO9249 NET                                       | Output ISO9249 NET                                       | Output ISO9249 NET                                       | PS (kW)/rpm                                              | 13.1 (9.6) / 2300                 |
|  5 |                                                          | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | 3                                 |
|  6 |                                                          | Bore × Stroke                                            | Bore × Stroke                                            | Bore × Stroke                                            | mm                                                       | 67 × 73.6                         |
|  7 |                                                          | Displacement                                             | Displacement                                             | Displacement                                             | cc                                                       | 778                               |
|  8 |                                                          | Overall width                                            | Overall width                                            | Overall width                                            | mm                                                       | 990                               |
|  9 |                                                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | mm                                                       | 2350 / 2330                       |
| 10 |                                                          | Overall length                                           | Overall length                                           | Overall length                                           | mm                                                       | 3710                              |
| 11 |                                                          | Ground clearance                                         | Ground clearance                                         | Ground clearance                                         | mm                                                       | 160                               |
| 12 | Dimensions                                               | Dozer size (width × height)                              | Dozer size (width × height)                              | Dozer size (width × height)                              | mm                                                       | 990 × 230                         |
| 13 |                                                          | Rubber shoe width                                        | Rubber shoe width                                        | Rubber shoe width                                        | mm                                                       | 230                               |
| 14 |                                                          | Minimum front swivel radius                              | Minimum front swivel radius                              | Minimum front swivel radius                              | mm                                                       | 1490                              |
| 15 |                                                          | Boom swing angle (left / right) deg                      | Boom swing angle (left / right) deg                      | Boom swing angle (left / right) deg                      | Boom swing angle (left / right) deg                      | 75 / 60                           |
| 16 | Hydraulic                                                | P1, P2                                                   |                                                          |                                                          |                                                          | Variable displacement pump        |
| 17 | Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 16.6 × 2                          |
| 18 | Hydraulic                                                |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.6 (210)                        |
| 19 | Hydraulic                                                | P3                                                       |                                                          |                                                          |                                                          | Gear pump                         |
| 20 | Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 10.4                              |
| 21 | System                                                   |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.1 (205)                        |
| 22 | Hydraulic                                                | Auxiliary                                                | /min Max. flow rate                                      | /min Max. flow rate                                      | /min Max. flow rate                                      | 27.0                              |
| 23 | Hydraulic                                                | (AUX)                                                    | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | 20.6 (210)                        |
| 24 | Hydraulic                                                | Max.                                                     | digging                                                  | arm                                                      | kN (kgf)                                                 | 7.3 (740)                         |
| 25 | Hydraulic                                                | force                                                    |                                                          | bucket                                                   | kN (kgf)                                                 | 12.7 (1300)                       |
| 26 | Hydraulic                                                | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | 28                                |
| 27 | Max. travelling speed km/h                               | Max. travelling speed km/h                               | Max. travelling speed km/h                               | Max. travelling speed km/h                               | Max. travelling speed km/h                               | 2.1                               |
| 28 | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | 25.5 (0.26) / 24.5 (0.25)         |
| 29 | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | 9.1                               |
| 30 | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | 21                                |
| 31 | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | 78                                |
| 32 | Noise level                                              | LwA                                                      |                                                          | (2000/14/EC)                                             | dB (A)                                                   | 93                                |
| 33 |                                                          | Hand arm (ISO5349-2:2001 )                               | system                                                   | Digging                                                  | m/s 2 RMS                                                | <2.5                              |
| 34 |                                                          |                                                          |                                                          | Levelling                                                | m/s 2 RMS                                                | <2.5                              |
| 35 |                                                          |                                                          |                                                          | Driving                                                  | m/s 2 RMS                                                | <2.5                              |
| 36 | Vibration* 3 KX015-4                                     |                                                          |                                                          | Idling                                                   | m/s 2 RMS                                                | <2.5                              |
| 37 |                                                          | Whole body (ISO2631- 1:1997) canopy                      |                                                          | Digging Lifting                                          | m/s 2 RMS m/s 2 RMS point radius                         | <0.5 <0.5 (2m)                    |
| 38 | Lift Point                                               | Height                                                   | Levelling                                                | Driving Over-front                                       | m/s 2 RMS                                                | <0.5                              |
| 39 |                                                          |                                                          | Blade                                                    | Idling Down                                              | m/s 2 RMS Blade UP                                       | <0.5 Over-side Blade              |

|    | KX015-4 cabin Cabin, Rubber version..Lift Point Height.   | Lifting point radius (2m).Over-front.Blade Down   | Lifting point radius (2m).Over-front.Blade UP   | Lifting point radius (2m).Over-side.   | Lifting point radius (max.).Over-front.Blade Down   | Lifting point radius (max.).Over-front.Blade UP   | kN (ton).Lifting point radius (max.).Over-side.Lift Point   |
|---:|:----------------------------------------------------------|:--------------------------------------------------|:------------------------------------------------|:---------------------------------------|:----------------------------------------------------|:--------------------------------------------------|:------------------------------------------------------------|
|  0 | 1.5m                                                      | 3.0 (0.30)                                        | 3.4 (0.35)                                      | 2.5 (0.26)                             | -                                                   | -                                                 | -                                                           |
|  1 | 1.0m                                                      | 4.5 (0.46)                                        | 3.5 (0.36)                                      | 2.3 (0.24)                             | -                                                   | -                                                 | -                                                           |
|  2 | 0.5m                                                      | 5.4 (0.55)                                        | 3.3 (0.34)                                      | 2.1 (0.22)                             | 2.6 (0.27)                                          | 1.6 (0.17)                                        | 1.1 (0.11) Lift Point Height                                |
|  3 | 0m                                                        | 5.3 (0.54)                                        | 3.2 (0.32)                                      | 2.0 (0.21)                             | -                                                   | -                                                 | -                                                           |

In [ ]:
lv1_cat, lv2_cat = "CE", "KOBUTA"
filename = "kubota_sample"
parsed_foldername = f"{lv1_cat}_{lv2_cat}_r02"
if not os.path.exists(f"./docs/{parsed_foldername}_r02"):
    os.makedirs(f"./docs/{parsed_foldername}_r02")
parsed_filename = filename.replace(".pdf", "")
with open(f"./docs/{parsed_foldername}/{parsed_filename}.pkl", 'ab') as file:
    pickle.dump(docs, file)